# 5100 HW1

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Check dataset
from pathlib import Path
p = Path("ml-100k")
assert p.exists() and any(p.iterdir()), "Place the 'ml-100k' folder next to this notebook."

# Load data
u_cols = ['user_id', 'age', 'sex', 'occupation', 'zip_code']
users  = pd.read_csv('ml-100k/u.user', sep='|', names=u_cols, encoding='latin-1')

r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=r_cols, encoding='latin-1')

m_cols = ['movie_id','title','release_date','video_release_date','imdb_url']
movies = pd.read_csv('ml-100k/u.item', sep='|', names=m_cols, usecols=range(5), encoding='latin-1')

def extract_year(s):
    if isinstance(s, str) and len(s)>=4:
        try:
            return int(s[-4:])
        except:
            return np.nan
    return np.nan

movies['year'] = movies['release_date'].apply(extract_year)

# Merge base
lens = ratings.merge(users, on='user_id').merge(movies, on='movie_id')

# Expand genres
g_cols = ['movie_id','title','release_date','video_release_date','imdb_url',
          'Unknown','Action','Adventure','Animation','Children\'s','Comedy','Crime','Documentary',
          'Drama','Fantasy','Film-Noir','Horror','Musical','Mystery','Romance','Sci-Fi','Thriller',
          'War','Western']
movies_g = pd.read_csv('ml-100k/u.item', sep='|', names=g_cols, encoding='latin-1')
genre_cols = g_cols[5:]

mg = movies_g.melt(id_vars=['movie_id','title'], value_vars=genre_cols,
                   var_name='genre', value_name='is_genre')
mg = mg[mg['is_genre']==1].drop(columns='is_genre')
lens_g = lens.merge(mg[['movie_id','genre']], on='movie_id', how='left')


## Q1

In [ ]:
'''[10 pts] The mean of movie ratings by men of age above 25 for each particular genre,
e.g., Action, Adventure, Drama, Science Fiction, ... Note, Action|Drama|Thriller’ is
not considered a unique genre. The movie that has a genre like this belongs to all three
genres.'''

men25 = lens_g[(lens_g['sex']=='M') & (lens_g['age']>25) & lens_g['genre'].notna()]
q1 = men25.groupby('genre')['rating'].mean().sort_values(ascending=False)
display(q1)

ax = q1.plot(kind='bar')
ax.set_title('Mean Ratings by Men >25 by Genre')
ax.set_ylabel('Mean rating'); ax.set_xlabel('Genre')
plt.tight_layout(); plt.show()


## Q2

In [ ]:
'''[5 pts] The top 5 ranked movies by the most number of ratings (not the highest rating).'''

q2 = lens.groupby('title').size().sort_values(ascending=False).head(5)
display(q2)


## Q3

In [ ]:
'''[20 pts] Average movie ratings between users of different age groups (<=18, 19-30,
31-50, 51-70, >=71)'''

bins = [-np.inf, 18, 30, 50, 70, np.inf]
labels = ['<=18', '19-30', '31-50', '51-70', '>=71']
lens['age_group5'] = pd.cut(lens['age'], bins=bins, labels=labels, right=True)

q3 = lens.groupby('age_group5')['rating'].agg(['size','mean']).rename(columns={'size':'num_ratings'})
display(q3)

ax = q3['mean'].plot(kind='bar')
ax.set_title('Average Rating by Age Group')
ax.set_ylabel('Mean rating'); ax.set_xlabel('Age group')
plt.tight_layout(); plt.show()


## Q4

In [ ]:
'''[25 pts] Pick a movie of your choice and for all movies of the same year, provide a
breakdown of the number of unique movies rated by 3 ranges of age of reviewers
(a) under 18 (b) 19 to 45 (c) Above 45.'''

selected_title = 'Toy Story (1995)'

sel_year = lens.loc[lens['title']==selected_title, 'year'].dropna().mode()
if len(sel_year)==0:
    raise ValueError('Year missing for the selected title.')
sel_year = int(sel_year.iloc[0])

same_year_movies = movies[movies['year']==sel_year][['movie_id','title','year']]
tmp = lens.merge(same_year_movies[['movie_id']], on='movie_id', how='inner')

def age_bucket3(a):
    if a < 18: return 'under18'
    if 19 <= a <= 45: return '19to45'
    return 'above45'
tmp['age_bucket3'] = tmp['age'].apply(age_bucket3)

q4 = tmp.groupby('age_bucket3')['movie_id'].nunique().reindex(['under18','19to45','above45'])
display({'year': sel_year, 'unique_movies': q4})

ax = q4.plot(kind='bar')
ax.set_title(f'Unique Movies Rated in {sel_year} by Age Buckets')
ax.set_ylabel('# Unique movies')
ax.set_xlabel('Age bucket')
plt.tight_layout()
plt.show()

Picked movie: Toy Story (1995).
Same-year unique titles rated: under-18 = 104, 19–45 = 217, above-45 = 181.

- 19–45 covers the widest set (217), ≈2.1× under-18 (104) and ~1.2× above-45 (181).
- Under-18 shows the narrowest coverage.
- The 19–45 vs above-45 gap suggests broader participation in mid-age cohorts.

## Q5

In [ ]:
'''[20 pts] A function that takes in a user_id and a movie_id, and returns a list of all
the other movies that the user rated similarly to the given movie, i.e. with the same
rating. Demonstrate that your function works.'''

title_map = movies.set_index('movie_id')['title'].to_dict()

def same_rating_others(df, user_id, movie_id):
    r = df[df['user_id']==user_id]
    base = r[r['movie_id']==movie_id]
    if base.empty:
        return pd.DataFrame(columns=['movie_id','title','rating'])
    target = int(base.iloc[0]['rating'])
    out = r[(r['rating']==target) & (r['movie_id']!=movie_id)][['movie_id','rating']].drop_duplicates()
    out['title'] = out['movie_id'].map(title_map)
    return out[['movie_id','title','rating']].sort_values('title')

ex = lens.sample(1, random_state=42).iloc[0]
uid, mid = int(ex['user_id']), int(ex['movie_id'])
print(f"user={uid}, movie={title_map[mid]!r}, rating={int(ex['rating'])}")
display(same_rating_others(lens, uid, mid).head(15))


## Q6

In [ ]:
'''[20 pts] Some other statistic, figure, aggregate, or plot that you created using this
dataset, along with a short description of what interesting observations you derived
from it.'''

# Observation: Genre × Age-group mean ratings

lens_g['age_group5'] = pd.cut(lens_g['age'], bins=bins, labels=labels, right=True)
q6 = lens_g.pivot_table(index='genre', columns='age_group5', values='rating', aggfunc='mean')
display(q6.round(3))

ax = q6.plot(kind='bar', figsize=(10,5))
ax.set_title('Mean Ratings by Genre and Age Group')
ax.set_ylabel('Mean rating'); ax.set_xlabel('Genre')
plt.tight_layout(); plt.show()


Observation: Genre × age-group mean ratings.

- Means tend to increase with age across several genres (e.g., Western 3.70 → 5.00; Mystery 3.59 → 4.50; Comedy 3.35 → 3.96; Action 3.64 → 4.00).
- Some genres are consistently high across groups (e.g., Documentary around 3.66–3.84).
- The ≥71 cohort has very few total ratings, so spikes (e.g., Western = 5.00) should be treated cautiously.
